# Inferencia con el clasificador de pulsos LAGO

Este notebook carga el modelo MLP entrenado en `MLP_model.ipynb` y muestra:

1. Carga del modelo `.keras` + metadata (parámetros de pre-procesamiento).
2. Cómo aplicar el pre-procesamiento a una señal nueva y obtener una predicción.
3. **Caracterización visual de cada clase** — perfil mediano del pulso por cluster + estadísticas de features. Esto te permite interpretar qué tipo de pulso representa cada clase.
4. Demo de inferencia sobre pulsos aleatorios del dataset.
5. Tabla interpretativa para mapear `clase → fenómeno físico` (placeholder editable).

In [ ]:
import os
import sys
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from lago_data_model.data_utils import signal_to_vector, segments_to_matrix

## 1. Cargar modelo y metadata

In [ ]:
import tensorflow as tf
from tensorflow import keras

MODEL_DIR  = "models"
MODEL_PATH = os.path.join(MODEL_DIR, "mlp_signal_best.keras")
META_PATH  = os.path.join(MODEL_DIR, "mlp_signal_meta.json")

with open(META_PATH) as f:
    meta = json.load(f)

model = keras.models.load_model(MODEL_PATH, compile=False)
model.summary()

print("\nMetadata:")
print(json.dumps(meta, indent=2))

## 2. Función de predicción

Recibe una señal cruda (array 1D) o lista de señales y devuelve `(clase_predicha, probabilidades)`. El pre-procesamiento se aplica con los parámetros guardados en metadata, así garantizamos que train e inferencia usan exactamente la misma transformación.

In [ ]:
PRE = meta["preprocessing"]
CLASS_LABELS = meta["class_labels"]

def predict_pulse(signal_or_signals):
    """
    Acepta:
      - 1 señal: array 1D de samples crudos del pulso
      - N señales: lista o iterable de arrays 1D (longitudes pueden variar)
    Retorna:
      classes: array de shape (N,) con la clase predicha
      probas:  array de shape (N, num_classes) con probabilidades softmax
    """
    is_single = (
        isinstance(signal_or_signals, np.ndarray) and signal_or_signals.ndim == 1
    ) or (not hasattr(signal_or_signals, "__iter__"))
    sigs = [signal_or_signals] if is_single else list(signal_or_signals)

    X = np.stack([
        signal_to_vector(
            s,
            n_before=PRE["n_before"],
            n_after=PRE["n_after"],
            resample_len=PRE["signal_len"],
            normalize=PRE["normalize"],
            scale=PRE["scale"],
        )
        for s in sigs
    ])
    logits = model.predict(X, verbose=0)
    probas = tf.nn.softmax(logits).numpy()
    classes = probas.argmax(axis=1)

    if is_single:
        return int(classes[0]), probas[0]
    return classes, probas

## 3. Caracterización de cada clase

**Pregunta clave**: "¿qué tipo de pulso representa la clase k?". El KMeans no asigna etiquetas físicas; las clases son sólo agrupamientos por similitud de features. Para interpretarlas calculamos:

- **Perfil mediano**: forma típica del pulso por clase (mediana sample-by-sample tras alinear).
- **Estadísticas de features físicas**: depth, width, area, FWHM, rise/recovery time medios.

Estos dos artefactos son la huella de cada clase y permiten asignarle un nombre físico (electrón, muón, ruido, etc.) basado en conocimiento del dominio.

In [ ]:
# Datos del clustering: necesarios para caracterizar cada clase
with open("all_segments.pkl", "rb") as f:
    all_segments = pickle.load(f)
with open("kmeans_labels.pkl", "rb") as f:
    kmeans_labels = pickle.load(f)
kmeans_labels = np.asarray(kmeans_labels)

# Features ya calculadas (para estadísticas físicas por clase)
pulses_df = pd.read_csv("pulses.csv")
feature_cols = ["min_value", "baseline_local", "depth", "width_samples",
                "area_under_pulse", "fwhm_samples", "rise_time",
                "recovery_time", "snr_depth_over_mad"]

print("segments:", len(all_segments), "| labels:", kmeans_labels.shape, "| pulses_df:", pulses_df.shape)
assert len(all_segments) == len(kmeans_labels) == len(pulses_df), "Desalineación entre segments/labels/features"

In [ ]:
# Perfil mediano por clase (sobre la señal pre-procesada, igual que en entrenamiento)
def class_profile(class_id, n_sample=2000, rng_seed=0):
    idxs = np.where(kmeans_labels == class_id)[0]
    if len(idxs) == 0:
        return None
    rng = np.random.default_rng(rng_seed)
    sel = rng.choice(idxs, size=min(n_sample, len(idxs)), replace=False)
    X = segments_to_matrix(
        [all_segments[i] for i in sel],
        n_before=PRE["n_before"],
        n_after=PRE["n_after"],
        resample_len=PRE["signal_len"],
        normalize=PRE["normalize"],
        scale=PRE["scale"],
    )
    return {
        "median": np.median(X, axis=0),
        "q25":    np.quantile(X, 0.25, axis=0),
        "q75":    np.quantile(X, 0.75, axis=0),
        "n":      len(idxs),
    }

profiles = {c: class_profile(c) for c in CLASS_LABELS}

In [ ]:
# Plot lado a lado de los perfiles típicos
import math
n_cls = len(CLASS_LABELS)
ncols = min(3, n_cls)
nrows = math.ceil(n_cls / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3*nrows), sharey=True)
axes = np.atleast_1d(axes).ravel()
for i, c in enumerate(CLASS_LABELS):
    p = profiles[c]
    if p is None: 
        axes[i].axis("off"); continue
    ax = axes[i]
    ax.fill_between(np.arange(len(p["median"])), p["q25"], p["q75"], alpha=0.3, label="IQR")
    ax.plot(p["median"], lw=2, color="black", label="mediana")
    ax.set_title(f"Clase {c}  (n={p['n']:,})")
    ax.set_xlabel("muestra"); ax.set_ylabel("amplitud")
    ax.grid(True); ax.legend(fontsize=8)
for j in range(i+1, len(axes)):
    axes[j].axis("off")
plt.suptitle("Perfil característico por clase", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Estadísticas de features físicas por clase: la "firma física" del cluster
stats_df = pulses_df.assign(cluster=kmeans_labels).groupby("cluster")[feature_cols].median()
stats_df["n"] = pd.Series(kmeans_labels).value_counts().sort_index()
stats_df["pct"] = (stats_df["n"] / stats_df["n"].sum() * 100).round(2)
stats_df

## 4. Casos de inferencia

Tomamos algunos pulsos al azar, los pasamos al modelo y comparamos con su perfil característico de clase.

In [ ]:
rng = np.random.default_rng(42)
N_DEMO = 6
demo_idx = rng.choice(len(all_segments), size=N_DEMO, replace=False)

raw_signals = [all_segments[i]["values"] for i in demo_idx]
true_classes = kmeans_labels[demo_idx]

pred_classes, pred_probas = predict_pulse(raw_signals)

fig, axes = plt.subplots(N_DEMO, 2, figsize=(13, 2.4*N_DEMO))
for i, (idx, raw, true_c, pred_c, p) in enumerate(zip(demo_idx, raw_signals, true_classes, pred_classes, pred_probas)):
    # Señal cruda
    ax_raw = axes[i, 0]
    ax_raw.plot(raw, lw=1)
    ax_raw.set_title(f"#{idx} | señal cruda (len={len(raw)})")
    ax_raw.grid(True)
    
    # Probabilidades + comparación con perfil de la clase predicha
    ax_prob = axes[i, 1]
    bars = ax_prob.bar(CLASS_LABELS, p, color=["green" if c == pred_c else "steelblue" for c in CLASS_LABELS])
    ax_prob.set_ylim(0, 1)
    correct = "✓" if int(pred_c) == int(true_c) else "✗"
    ax_prob.set_title(f"pred={pred_c} (true={true_c}) {correct} | conf={p[pred_c]:.2f}")
    ax_prob.set_xlabel("clase"); ax_prob.set_ylabel("P")
    ax_prob.grid(True, axis="y")

plt.tight_layout(); plt.show()

## 5. Tabla interpretativa

Combinando el perfil mediano + estadísticas de features podés mapear cada clase del KMeans a un tipo físico de pulso. **Editá los `tipo` y `notas` abajo** según tu conocimiento del dominio (e.g., muones tienen `depth` alto y `width` chico; ruido suele tener `snr` bajo, etc.).

Una vez asignados, esta tabla es la "leyenda" del modelo y deberías guardarla junto a `mlp_signal_meta.json` o, mejor aún, pasarla al campo `class_names` de la metadata.

In [ ]:
# Plantilla: completar 'tipo' y 'notas' a mano según las firmas observadas
interp = pd.DataFrame({
    "clase": CLASS_LABELS,
    "n":     [profiles[c]["n"] for c in CLASS_LABELS],
    "depth_med":          stats_df.loc[CLASS_LABELS, "depth"].values,
    "width_med":          stats_df.loc[CLASS_LABELS, "width_samples"].values,
    "fwhm_med":           stats_df.loc[CLASS_LABELS, "fwhm_samples"].values,
    "snr_med":            stats_df.loc[CLASS_LABELS, "snr_depth_over_mad"].values,
    "rise_time_med":      stats_df.loc[CLASS_LABELS, "rise_time"].values,
    "tipo":               ["?"] * len(CLASS_LABELS),   # ← editar a mano
    "notas":              [""] * len(CLASS_LABELS),    # ← editar a mano
})
interp

In [ ]:
# Una vez que llenes 'tipo' y 'notas', guardalo:
# interp.to_csv("models/class_interpretation.csv", index=False)
# Y/o agregalo a la metadata:
# meta['class_names'] = dict(zip(interp['clase'], interp['tipo']))
# with open(META_PATH, 'w') as f: json.dump(meta, f, indent=2)
pass

## 6. Pipeline end-to-end de un pulso nuevo

Función única que recibe una señal cruda y devuelve clase + tipo (si la metadata tiene `class_names`).

In [ ]:
def classify(raw_signal):
    """Pipeline completo: señal cruda → (clase_id, tipo_físico, prob)"""
    cls, prob = predict_pulse(raw_signal)
    tipo = meta.get("class_names", {}).get(str(cls), "sin etiqueta")
    return {"class_id": int(cls), "tipo": tipo, "confidence": float(prob[cls]), "probas": prob.tolist()}

# ejemplo
sample = all_segments[0]["values"]
result = classify(sample)
print("Pulso #0 →", result)